In [4]:
import numpy as np
import polars as pl

In [2]:
pl.__version__

'1.8.2'

HF_ENDPOINT="http://huggingface.proxy" hf download deepvk/VK-LSVD --repo-type dataset --include "metadata/*" --local-dir  /home/jovyan/IRec/sigir/lsvd_data/raw

HF_ENDPOINT="http://huggingface.proxy" hf download deepvk/VK-LSVD --repo-type dataset --include "subsamples/ur0.01_ir0.01/*" --local-dir  /home/jovyan/IRec/sigir/lsvd_data/raw


Разбиение сабсэмплов на базовую, гэп, вал и тест части

Добавляется колонка original_order чтобы сохранять порядок внутри каждой из частей

In [ ]:
subsample_name = 'ur0.01_ir0.01'
content_embedding_size = 256
DATASET_PATH = "/home/jovyan/IRec/sigir/lsvd_data_filtered/raw"

metadata_files = ['metadata/users_metadata.parquet',
                  'metadata/items_metadata.parquet',
                  'metadata/item_embeddings.npz']

BASE_WEEKS = (0, 22)
GAP_WEEKS = (22, 23) #увеличить гэп
VAL_WEEKS = (23, 24)

base_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(BASE_WEEKS[0], BASE_WEEKS[1])]

gap_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(GAP_WEEKS[0], GAP_WEEKS[1])]

val_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(VAL_WEEKS[0], VAL_WEEKS[1])]

test_interactions_files = [f'subsamples/{subsample_name}/train/week_24.parquet']

all_interactions_files = base_interactions_files + gap_interactions_files + val_interactions_files + test_interactions_files

base_with_gap_interactions_files = base_interactions_files + gap_interactions_files

len(base_interactions_files), len(gap_interactions_files), len(val_interactions_files), len(test_interactions_files), len(all_interactions_files), len(base_with_gap_interactions_files)

(22, 1, 1, 1, 25, 23)

In [13]:
base_interactions_files[-1], gap_interactions_files, val_interactions_files, test_interactions_files

('subsamples/ur0.01_ir0.01/train/week_21.parquet',
 ['subsamples/ur0.01_ir0.01/train/week_22.parquet'],
 ['subsamples/ur0.01_ir0.01/train/week_23.parquet'],
 ['subsamples/ur0.01_ir0.01/train/week_24.parquet'])

In [16]:
def get_parquet_interactions(data_files, positive_event_timespent):
    data_interactions = pl.concat([pl.scan_parquet(f'{DATASET_PATH}/{file}')
                                for file in data_files])
    data_interactions = data_interactions.collect(streaming=True)
    data_interactions = data_interactions.with_row_index("original_order")
    data_interactions = data_interactions.filter(pl.col('timespent') > positive_event_timespent)
    return data_interactions


In [17]:
POSITIVE_EVENT_TIMESPENT = 15
base_interactions = get_parquet_interactions(base_interactions_files, POSITIVE_EVENT_TIMESPENT)
gap_interactions = get_parquet_interactions(gap_interactions_files, POSITIVE_EVENT_TIMESPENT)
val_interactions = get_parquet_interactions(val_interactions_files, POSITIVE_EVENT_TIMESPENT)
test_interactions = get_parquet_interactions(test_interactions_files, POSITIVE_EVENT_TIMESPENT)
all_data_interactions = get_parquet_interactions(all_interactions_files, POSITIVE_EVENT_TIMESPENT)
base_with_gap_interactions = get_parquet_interactions(base_with_gap_interactions_files, POSITIVE_EVENT_TIMESPENT)

Загрузка и фильтрация эмбеддингов

In [19]:
all_data_users = all_data_interactions.select('user_id').unique()
all_data_items = all_data_interactions.select('item_id').unique()

item_ids = np.load(f"{DATASET_PATH}/metadata/item_embeddings.npz")['item_id']
item_embeddings = np.load(f"{DATASET_PATH}/metadata/item_embeddings.npz")['embedding']

mask = np.isin(item_ids, all_data_items.to_numpy())
item_ids = item_ids[mask]
item_embeddings = item_embeddings[mask]
item_embeddings = item_embeddings[:, :content_embedding_size]

users_metadata = pl.read_parquet(f"{DATASET_PATH}/metadata/users_metadata.parquet")
items_metadata = pl.read_parquet(f"{DATASET_PATH}/metadata/items_metadata.parquet")

users_metadata = users_metadata.join(all_data_users, on='user_id')
items_metadata = items_metadata.join(all_data_items, on='item_id')
items_metadata = items_metadata.join(pl.DataFrame({'item_id': item_ids, 
                                                   'embedding': item_embeddings}), on='item_id')

only_base_items_metadata = items_metadata.join(base_interactions.select('item_id').unique(), on='item_id')
only_base_with_gap_items_metadata = items_metadata.join(base_with_gap_interactions.select('item_id').unique(), on='item_id')

Сжатие айтем айди и ремапинг

In [20]:
all_data_items = all_data_interactions.select('item_id').unique()
all_data_users = all_data_interactions.select('user_id').unique()

unique_items_sorted = all_data_items.sort('item_id').with_row_index('new_item_id')
global_item_mapping = dict(zip(unique_items_sorted['item_id'], unique_items_sorted['new_item_id']))

print(f"Total users: {all_data_users.shape[0]}, Total items: {len(global_item_mapping)}")

Total users: 77403, Total items: 65659


In [21]:
def remap_interactions(df, mapping):
    return df.with_columns(
        pl.col('item_id')
        .map_elements(lambda x: mapping.get(x, None), return_dtype=pl.UInt32)
    )

base_interactions_remapped = remap_interactions(base_interactions, global_item_mapping)
gap_interactions_remapped = remap_interactions(gap_interactions, global_item_mapping)
test_interactions_remapped = remap_interactions(test_interactions, global_item_mapping)
val_interactions_remapped = remap_interactions(val_interactions, global_item_mapping)
all_data_interactions_remapped = remap_interactions(all_data_interactions, global_item_mapping)

In [22]:
del base_interactions, gap_interactions, test_interactions, val_interactions, all_data_interactions

In [23]:
base_with_gap_interactions_remapped = remap_interactions(base_with_gap_interactions, global_item_mapping)
del base_with_gap_interactions

In [24]:
items_metadata_remapped = remap_interactions(items_metadata, global_item_mapping)
only_base_items_metadata_remapped = remap_interactions(only_base_items_metadata, global_item_mapping)
only_base_with_gap_items_metadata_remapped = remap_interactions(only_base_with_gap_items_metadata, global_item_mapping)

Группировка по юзер айди

In [25]:
def get_grouped_interactions(data_interactions):
    print(f"interactions count: {data_interactions.shape}")
    data_res = (
        data_interactions
        .select(['original_order', 'user_id', 'item_id'])
        .group_by('user_id')
        .agg(
            pl.col('item_id')
            .sort_by(pl.col('original_order'))
            .alias('item_ids'),
            pl.col('original_order').alias('timestamps')
        )
        .rename({'user_id': 'uid'})
    )
    print(f"users count: {data_res.shape}")
    return data_res
base_interactions_grouped = get_grouped_interactions(base_interactions_remapped)
gap_interactions_grouped = get_grouped_interactions(gap_interactions_remapped)
test_interactions_grouped = get_grouped_interactions(test_interactions_remapped)
val_interactions_grouped = get_grouped_interactions(val_interactions_remapped)
all_data_interactions_grouped = get_grouped_interactions(all_data_interactions_remapped)

interactions count: (1206762, 13)
users count: (75281, 3)
interactions count: (66879, 13)
users count: (28610, 3)
interactions count: (69887, 13)
users count: (28982, 3)
interactions count: (73637, 13)
users count: (30231, 3)
interactions count: (1417165, 13)
users count: (77403, 3)


In [26]:
base_interactions_grouped.head(1)

uid,item_ids,timestamps
u32,list[u32],list[u32]
4651912,"[39251, 11631, … 18532]","[181275, 1237407, … 3130963]"


In [27]:
del base_interactions_remapped, gap_interactions_remapped, test_interactions_remapped, val_interactions_remapped

In [ ]:
base_with_gap_interactions_grouped = get_grouped_interactions(base_with_gap_interactions_remapped)
del base_with_gap_interactions_remapped

interactions count: (1273641, 13)
users count: (76011, 3)


Сохранение

In [29]:
import json
OUTPUT_DIR = "/home/jovyan/IRec/sigir/lsvd_data_filtered/15-ts-ows"

mapping_output_path = f"{OUTPUT_DIR}/global_item_mapping.json"

with open(mapping_output_path, 'w') as f:
    json.dump({str(k): v for k, v in global_item_mapping.items()}, f, indent=2)

print(f"Сохранён маппинг: {mapping_output_path}")

Сохранён маппинг: /home/jovyan/IRec/sigir/lsvd_data_filtered/15-ts-ows/global_item_mapping.json


In [30]:
def write_parquet(output_dir, data, file_name):
    print(f"размерность: {data.shape}")
    output_parquet_path = f"{output_dir}/{file_name}.parquet"
    data.write_parquet(output_parquet_path)
    print(f"Сохранен файл: {file_name}")

write_parquet(OUTPUT_DIR, items_metadata_remapped, "items_metadata_remapped")
write_parquet(OUTPUT_DIR, items_metadata, "items_metadata_old")

write_parquet(OUTPUT_DIR, only_base_items_metadata_remapped, "only_base_items_metadata_remapped")
write_parquet(OUTPUT_DIR, only_base_items_metadata, "only_base_items_metadata_old")

write_parquet(OUTPUT_DIR, only_base_with_gap_items_metadata_remapped, "only_base_with_gap_items_metadata_remapped")
write_parquet(OUTPUT_DIR, only_base_with_gap_items_metadata, "only_base_with_gap_items_metadata_old")

write_parquet(OUTPUT_DIR, base_interactions_grouped, "base_interactions_grouped")
write_parquet(OUTPUT_DIR, gap_interactions_grouped, "gap_interactions_grouped")
write_parquet(OUTPUT_DIR, test_interactions_grouped, "test_interactions_grouped")
write_parquet(OUTPUT_DIR, val_interactions_grouped, "val_interactions_grouped")
write_parquet(OUTPUT_DIR, base_with_gap_interactions_grouped, "base_with_gap_interactions_grouped")

write_parquet(OUTPUT_DIR, all_data_interactions_grouped, "all_data_interactions_grouped")

write_parquet(OUTPUT_DIR, all_data_interactions_remapped, "all_data_interactions_remapped")

размерность: (65659, 5)
Сохранен файл: items_metadata_remapped
размерность: (65659, 5)
Сохранен файл: items_metadata_old
размерность: (60552, 5)
Сохранен файл: only_base_items_metadata_remapped
размерность: (60552, 5)
Сохранен файл: only_base_items_metadata_old
размерность: (62362, 5)
Сохранен файл: only_base_with_gap_items_metadata_remapped
размерность: (62362, 5)
Сохранен файл: only_base_with_gap_items_metadata_old
размерность: (75281, 3)
Сохранен файл: base_interactions_grouped
размерность: (28610, 3)
Сохранен файл: gap_interactions_grouped
размерность: (28982, 3)
Сохранен файл: test_interactions_grouped
размерность: (30231, 3)
Сохранен файл: val_interactions_grouped
размерность: (76011, 3)
Сохранен файл: base_with_gap_interactions_grouped
размерность: (77403, 3)
Сохранен файл: all_data_interactions_grouped
размерность: (1417165, 13)
Сохранен файл: all_data_interactions_remapped


In [31]:
len(list(items_metadata_remapped.head(1)['embedding'].item())), len(list(items_metadata.head(1)['embedding'].item()))

(64, 64)

In [33]:
items_metadata_remapped.shape, only_base_items_metadata_remapped.shape, only_base_with_gap_items_metadata_remapped.shape

((65659, 5), (60552, 5), (62362, 5))

In [34]:
items_metadata_remapped.head(1)

item_id,author_id,duration,train_interactions_rank,embedding
u32,u32,u8,u32,"array[f32, 64]"
0,651705,71,14592477,"[-0.281006, -0.145508, … -0.058258]"


In [35]:
base_with_gap_interactions_grouped.head()

uid,item_ids,timestamps
u32,list[u32],list[u32]
1450406,"[65092, 7570]","[2309584, 3257670]"
3736304,"[39094, 42731, … 10818]","[2691900, 2743200, … 3232861]"
775866,"[52462, 44617, … 64047]","[101336, 161036, … 2789649]"
777572,"[48639, 6069]","[1056275, 1218815]"
1942489,"[44575, 6264, 32234]","[372244, 955295, 2432349]"
